# Mininet PCAP Training - Single File Upload

**Simplified for Colab:**
- Upload ONE combined PCAP file
- Automatic attack detection based on traffic patterns
- Complete training + prediction pipeline
- Comprehensive visualizations

In [ ]:
# Install dependencies
!pip install -q scapy pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn
print("✓ Dependencies installed")

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict
import joblib

from google.colab import files
from scapy.all import rdpcap, IP, TCP, UDP, ICMP

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE

print("✓ Libraries imported")

## Upload Combined PCAP File

In [ ]:
print("Upload your combined PCAP file...")
print("(Should contain both normal and attack traffic)\n")

uploaded = files.upload()
pcap_file = list(uploaded.keys())[0]

print(f"\n✓ Uploaded: {pcap_file} ({len(uploaded[pcap_file])} bytes)")

## Feature Extraction with Intelligent Labeling

In [ ]:
class IntelligentPCAPExtractor:
    """Extract features and intelligently label traffic"""
    
    def detect_attack_type(self, flow_features):
        """Detect if flow is attack based on characteristics"""
        
        # SYN Flood detection
        if (flow_features['syn_ratio'] > 0.8 and 
            flow_features['packets_per_sec'] > 100):
            return 1, 'syn_flood'
        
        # Port Scan detection
        if (flow_features['packet_count'] < 5 and
            flow_features['rst_ratio'] > 0.5):
            return 1, 'port_scan'
        
        # UDP Flood detection
        if (flow_features['protocol'] == 'UDP' and
            flow_features['packets_per_sec'] > 50):
            return 1, 'udp_flood'
        
        # HTTP Flood detection
        if (flow_features['dst_port'] == 80 and
            flow_features['packets_per_sec'] > 50):
            return 1, 'http_flood'
        
        # Normal traffic
        return 0, 'normal'
    
    def extract_from_pcap(self, pcap_file):
        """Extract and label features from PCAP"""
        print(f"\nProcessing: {pcap_file}")
        
        try:
            packets = rdpcap(pcap_file)
            print(f"  Total packets: {len(packets)}")
        except Exception as e:
            print(f"  ❌ Error: {e}")
            return []
        
        # Group by flow
        flows = defaultdict(list)
        for pkt in packets:
            if IP in pkt:
                flow_key = self._get_flow_key(pkt)
                if flow_key:
                    flows[flow_key].append(pkt)
        
        print(f"  Flows identified: {len(flows)}")
        
        # Extract features
        features = []
        for flow_key, flow_packets in flows.items():
            feature = self._extract_flow_features(flow_key, flow_packets)
            if feature:
                # Intelligent labeling
                label, attack_type = self.detect_attack_type(feature)
                feature['label'] = label
                feature['attack_type'] = attack_type
                features.append(feature)
        
        print(f"  ✓ Extracted {len(features)} flows")
        
        # Show distribution
        normal = sum(1 for f in features if f['label'] == 0)
        attack = sum(1 for f in features if f['label'] == 1)
        print(f"  Normal: {normal}, Attack: {attack}")
        
        return features
    
    def _get_flow_key(self, pkt):
        if IP not in pkt:
            return None
        
        src_ip, dst_ip = pkt[IP].src, pkt[IP].dst
        
        if TCP in pkt:
            return (src_ip, dst_ip, pkt[TCP].sport, pkt[TCP].dport, 'TCP')
        elif UDP in pkt:
            return (src_ip, dst_ip, pkt[UDP].sport, pkt[UDP].dport, 'UDP')
        elif ICMP in pkt:
            return (src_ip, dst_ip, 0, 0, 'ICMP')
        return None
    
    def _extract_flow_features(self, flow_key, packets):
        src_ip, dst_ip, src_port, dst_port, protocol = flow_key
        
        if len(packets) == 0:
            return None
        
        timestamps = [float(pkt.time) for pkt in packets]
        duration = max(timestamps) - min(timestamps) if len(timestamps) > 1 else 0.001
        
        packet_sizes = [len(pkt) for pkt in packets]
        packet_count = len(packets)
        byte_count = sum(packet_sizes)
        
        syn_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x02)
        fin_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x01)
        rst_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x04)
        psh_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x08)
        ack_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x10)
        
        packets_per_sec = packet_count / duration
        bytes_per_sec = byte_count / duration
        
        if len(timestamps) > 1:
            iat = np.diff(timestamps)
            mean_iat, std_iat = np.mean(iat), np.std(iat)
        else:
            mean_iat, std_iat = 0, 0
        
        return {
            'duration': duration,
            'protocol': protocol,
            'src_port': src_port,
            'dst_port': dst_port,
            'packet_count': packet_count,
            'byte_count': byte_count,
            'packets_per_sec': packets_per_sec,
            'bytes_per_sec': bytes_per_sec,
            'mean_packet_size': np.mean(packet_sizes),
            'std_packet_size': np.std(packet_sizes) if len(packet_sizes) > 1 else 0,
            'min_packet_size': min(packet_sizes),
            'max_packet_size': max(packet_sizes),
            'mean_inter_arrival_time': mean_iat,
            'std_inter_arrival_time': std_iat,
            'syn_count': syn_count,
            'fin_count': fin_count,
            'rst_count': rst_count,
            'psh_count': psh_count,
            'ack_count': ack_count,
            'syn_ratio': syn_count / packet_count,
            'fin_ratio': fin_count / packet_count,
            'rst_ratio': rst_count / packet_count,
            'psh_ratio': psh_count / packet_count,
            'ack_ratio': ack_count / packet_count,
            'is_well_known_port': 1 if dst_port < 1024 else 0
        }

print("✓ Extractor defined")

## Process PCAP File

In [ ]:
print("="*60)
print("PROCESSING PCAP FILE")
print("="*60)

extractor = IntelligentPCAPExtractor()
features = extractor.extract_from_pcap(pcap_file)

df = pd.DataFrame(features)

print(f"\n{'='*60}")
print(f"✓ Total samples: {len(df):,}")
print(f"  Normal: {len(df[df['label'] == 0]):,}")
print(f"  Attack: {len(df[df['label'] == 1]):,}")
print("="*60)

# Check if we have both classes
if len(df['label'].unique()) < 2:
    print("\n⚠ WARNING: Only one class detected!")
    print("The PCAP file may contain only normal or only attack traffic.")
    print("Please upload a combined PCAP with both types.")
else:
    print("\n✓ Both classes present - ready for training!")

df.head()

## Data Preprocessing

In [ ]:
# Separate features and labels
X = df.drop(['label', 'attack_type'], axis=1)
y = df['label']

# Encode categorical
label_encoders = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

X = X.fillna(0).replace([np.inf, -np.inf], 0)

print(f"✓ Features: {len(X.columns)}, Samples: {len(X)}")

## Train/Val/Test Split & Feature Engineering

In [ ]:
# Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Feature selection
k_features = min(25, X_train.shape[1])
selector = SelectKBest(mutual_info_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"✓ Selected {len(selected_features)} features")

# SMOTE only if we have both classes
if len(np.unique(y_train)) > 1:
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
    print(f"✓ Balanced: {len(X_train_balanced)} samples")
else:
    print("⚠ Skipping SMOTE - only one class in training data")
    X_train_balanced, y_train_balanced = X_train_selected, y_train

In [ ]:
# Continue with rest of training...
# (Same as before: Train models, evaluate, visualize, save, predict)